In [0]:
%restart_python

In [0]:
from pathlib import Path
import sys
import os


def is_databricks():
    return "DATABRICKS_RUNTIME_VERSION" in os.environ

if is_databricks():
    sys.path.append(str(Path.cwd().parent / 'src'))

In [0]:
import mlflow
import optuna
import pandas as pd
import numpy as np

from config import Tags, ProjectConfig
from pyspark.sql import SparkSession
from loguru import logger
from delta.tables import DeltaTable

from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.metrics import f1_score


config = ProjectConfig.from_yaml(config_path="../project_config_telcochurn.yml", env="dev")
spark = SparkSession.builder.getOrCreate()
tags = Tags(**{"git_sha": "abcd12345", "branch": "main"})


class HyperParameterTuning:

    def __init__(self, config: ProjectConfig, tags: Tags, spark: SparkSession) -> None:
        self.config = config
        self.spark = spark
        self.num_features = self.config.num_features
        self.cat_features = self.config.cat_features
        self.target = self.config.target
        self.parameters = self.config.parameters
        self.catalog_name = self.config.catalog_name
        self.schema_name = self.config.schema_name
        self.experiment_name = self.config.experiment_name_basic
        self.model_name = f"{self.catalog_name}.{self.schema_name}.telco_churn_lightgbm_model"
        self.tags = tags.to_dict()

    def load_data(self) -> None:
        logger.info("Loading data from Databricks tables...")

        self.train_set_spark = self.spark.table(f"{self.catalog_name}.{self.schema_name}.telco_features_train")
        self.train_set = self.train_set_spark.toPandas()
        self.test_set_spark = self.spark.table(f"{self.catalog_name}.{self.schema_name}.telco_features_test")
        self.test_set = self.test_set_spark.toPandas()

        features = self.num_features + self.cat_features

        if "customerID" in features:
            features.remove("customerID")

        self.cat_features = [col for col in self.cat_features if col in features]

        for col in self.cat_features:
            self.train_set[col] = self.train_set[col].astype("category")
            self.test_set[col] = self.test_set[col].astype("category")

        self.X_train = self.train_set[features]
        self.X_test = self.test_set[features]

        self.y_train = self.train_set[self.target]
        self.y_test = self.test_set[self.target]

        if self.y_train.dtype == "object":
            self.y_train = self.y_train.map({"No": 0, "Yes": 1}).astype(int)
            self.y_test = self.y_test.map({"No": 0, "Yes": 1}).astype(int)

        object_columns = self.X_train.select_dtypes(include="object").columns.tolist()

        if len(object_columns) > 0:
            raise ValueError(f"Object columns found in X_train: {object_columns}. Remove them or convert them to category.")

        train_delta_table = DeltaTable.forName(self.spark, f"{self.catalog_name}.{self.schema_name}.telco_features_train")
        self.train_data_version = str(train_delta_table.history().select("version").first()[0])

        test_delta_table = DeltaTable.forName(self.spark, f"{self.catalog_name}.{self.schema_name}.telco_features_test")
        self.test_data_version = str(test_delta_table.history().select("version").first()[0])

        logger.info("Data successfully loaded.")
        logger.info(f"Train data version: {self.train_data_version}")
        logger.info(f"Test data version: {self.test_data_version}")
        logger.info(f"Features used: {features}")
        logger.info(f"Categorical features used: {self.cat_features}")

    def objective(self, trial: optuna.Trial) -> float:
        params = {
            "objective": "binary",
            "n_estimators": 5000,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,
        }

        threshold = trial.suggest_float("threshold", 0.2, 0.8)

        logger.info("Training LightGBM model...")

        model = LGBMClassifier(**params)

        model.fit(
            self.X_train,
            self.y_train,
            eval_set=[(self.X_test, self.y_test)],
            eval_metric="binary_logloss",
            categorical_feature=self.cat_features,
            callbacks=[
                early_stopping(stopping_rounds=100),
                log_evaluation(period=0),
            ],
        )

        y_pred_proba = model.predict_proba(self.X_test)[:, 1]
        y_pred = (y_pred_proba >= threshold).astype(int)

        f1 = f1_score(self.y_test, y_pred)

        logger.info(f"Trial F1-score: {f1}")
        logger.info(f"Trial threshold: {threshold}")

        return f1

    def run_tuning(self, n_trials: int = 50) -> optuna.Study:
        logger.info("Starting hyperparameter tuning...")

        study = optuna.create_study(direction="maximize")

        study.optimize(
            self.objective,
            n_trials=n_trials,
        )

        logger.info(f"Best F1-score: {study.best_value}")
        logger.info(f"Best parameters: {study.best_params}")

        print("Melhor F1-score:", study.best_value)
        print("Melhores parâmetros:", study.best_params)

        return study

    def train_best_model(self, study: optuna.Study) -> LGBMClassifier:
        best_params = study.best_params.copy()
        self.best_threshold = best_params.pop("threshold")

        final_params = {
            **best_params,
            "objective": "binary",
            "n_estimators": 5000,
            "random_state": 42,
            "n_jobs": -1,
            "verbosity": -1,
        }

        logger.info("Training final LightGBM model with best parameters...")

        self.final_model = LGBMClassifier(**final_params)

        self.final_model.fit(
            self.X_train,
            self.y_train,
            eval_set=[(self.X_test, self.y_test)],
            eval_metric="binary_logloss",
            categorical_feature=self.cat_features,
            callbacks=[
                early_stopping(stopping_rounds=100),
                log_evaluation(period=100),
            ],
        )

        y_pred_proba = self.final_model.predict_proba(self.X_test)[:, 1]
        y_pred = (y_pred_proba >= self.best_threshold).astype(int)

        final_f1 = f1_score(self.y_test, y_pred)

        logger.info(f"Best threshold: {self.best_threshold}")
        logger.info(f"Final F1-score: {final_f1}")

        print("Best threshold:", self.best_threshold)
        print("Final F1-score:", final_f1)

        return self.final_model


tuning = HyperParameterTuning(config=config, tags=tags, spark=spark)

tuning.load_data()

study = tuning.run_tuning(n_trials=50)

final_model = tuning.train_best_model(study)

print(tuning.model_name)